# needle-fp4: FP4 training experiments on Colab

Runs the Stage 2 tests (including the CUDA versions) and the Stage 3 MNIST experiments.

**Before running:** Runtime > Change runtime type > **GPU**. Then Runtime > Run all.

What it does:
1. Clones (or updates) the repo into `MyDrive/10714/fp4-needle`, so results survive the session.
2. Builds needle's CPU and CUDA backends with `make`.
3. Runs `tests/fp4` on every device, including the CUDA bit-exactness tests that cannot run without a GPU.
4. Trains the MLP on MNIST for every FP4 configuration and plots the results.

In [ ]:
# Which branch to run (a dev/<name>-<topic> branch to try unmerged work).
BRANCH = "main"
SEEDS = [0, 1, 2]
EPOCHS = 3
DEVICE = "cuda"   # switch to "cpu" if the GPU turns out slower (see the timing cell)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
%cd /content/drive/MyDrive/10714
!test -d fp4-needle || git clone https://github.com/shreya-v7/fp4-needle.git
%cd /content/drive/MyDrive/10714/fp4-needle
!git fetch origin && git checkout {BRANCH} && git pull --ff-only origin {BRANCH}
!git log --oneline -3

In [ ]:
!pip3 install -q pybind11
!make clean && make
%set_env PYTHONPATH ./python
%set_env NEEDLE_BACKEND nd
!nvidia-smi --query-gpu=name,memory.total --format=csv
!python3 -c "import sys; sys.path.append('python'); import needle as ndl; print('CUDA backend built:', ndl.cuda().enabled())"

## Tests
All of `tests/fp4` should pass. On a GPU the `cuda()` variants run too: they check that the
new `floor` / `abs` kernels work and that FP4 fake quantization on the GPU is bit-identical
to the NumPy reference.

In [ ]:
!python3 -m pytest tests/fp4 -q

## MNIST data
Copies MNIST from our homework folder (`MyDrive/10714/hw2/data`) into `data/` (not tracked by git).

In [ ]:
!mkdir -p data && cp -n ../hw2/data/*-ubyte.gz data/ && ls -la data

## Timing check
Needle's CUDA matmul uses 4x4 tiles and fake quantization launches many small kernels, so the
GPU is not guaranteed to beat the CPU. This times a few steps on each device.

In [ ]:
import sys, time
sys.path.append('python'); sys.path.append('apps')
import numpy as np, needle as ndl
import fp4_experiment as E
from needle.fp4.config import QuantConfig

train, _ = E.load_mnist('data', 1280)
for dev_name in ['cpu', 'cuda']:
    dev = getattr(ndl, dev_name)()
    for cfg in [QuantConfig(), QuantConfig(fprop=True, dgrad=True, wgrad=True, stochastic=True)]:
        np.random.seed(0)
        model = E.make_mlp(cfg, 256, dev)
        opt = ndl.optim.SGD(model.parameters(), lr=0.05, momentum=0.9)
        loss_fn = ndl.nn.SoftmaxLoss()
        start, steps = time.time(), 0
        for batch in ndl.data.DataLoader(train, batch_size=128, shuffle=True):
            X, y = E.to_device(batch, dev)
            opt.reset_grad(); loss = loss_fn(model(X), y); loss.backward(); opt.step()
            float(loss.numpy())  # forces the GPU to finish this step
            steps += 1
        print(f"{dev_name:5s} {cfg.name:22s} {(time.time() - start) / steps * 1000:6.0f} ms/step")

## Experiment 1: the project grid plus single-matmul ablations (SGD)
10 configurations x 3 seeds. Each run uses the same seed for every config, so runs differ only by FP4.

In [ ]:
for seed in SEEDS:
    !python3 apps/fp4_experiment.py --configs all --device {DEVICE} --epochs {EPOCHS} --seed {seed} --out results/colab_sgd_seed{seed}.csv
!python3 apps/plot_fp4.py results/colab_sgd_seed*.csv --out-dir results/colab_sgd

In [ ]:
from IPython.display import Image, Markdown, display
display(Image('results/colab_sgd/mnist_grid.png'))
display(Image('results/colab_sgd/mnist_ablation.png'))
display(Markdown(open('results/colab_sgd/summary.md').read()))

## Experiment 2: same grid with Adam
Adam adapts the step size per parameter, so it reacts differently to noisy or biased FP4 gradients.

In [ ]:
for seed in SEEDS:
    !python3 apps/fp4_experiment.py --configs all --device {DEVICE} --epochs {EPOCHS} --seed {seed} --optimizer adam --lr 0.001 --out results/colab_adam_seed{seed}.csv
!python3 apps/plot_fp4.py results/colab_adam_seed*.csv --out-dir results/colab_adam
display(Image('results/colab_adam/mnist_grid.png'))
display(Image('results/colab_adam/mnist_ablation.png'))
display(Markdown(open('results/colab_adam/summary.md').read()))

Results (CSVs, plots, `summary.md`) are in `MyDrive/10714/fp4-needle/results/` on Drive.